In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")          # pour que Python trouve le dossier src/

import pandas as pd
import matplotlib.pyplot as plt
from src.donnees import charger_raca

In [2]:
raca = charger_raca()
print(raca.shape)
help(charger_raca)

(188711, 29)
Help on function charger_raca in module src.donnees:

charger_raca(
    dossier=WindowsPath('C:/Users/FAYA COMPUTER/Documents/PROJET SMART ICT S5/5g6g-perf-anomaly-ml-dl/data/raw/raca5g/5G-production-dataset')
)
    Charge les 83 traces du jeu 5G de Raca et al. dans un seul DataFrame.

    Trois colonnes sont ajoutées pour garder l'origine de chaque ligne :
    application (Netflix, Amazon_Prime, Download), mobilite (Static, Driving)
    et trace (nom du fichier).



In [3]:
manquantes = raca.isna().mean().sort_values(ascending=False)
manquantes[manquantes > 0].round(3)

PINGMIN      0.983
PINGMAX      0.983
PINGSTDEV    0.983
PINGAVG      0.983
PINGLOSS     0.983
RSSI         0.284
NRxRSRQ      0.197
NRxRSRP      0.183
CQI          0.091
SNR          0.091
RSRQ         0.019
NODEHEX      0.001
dtype: float64

In [4]:
print("RSRP = -200 :", (raca["RSRP"] == -200).sum())
raca[raca["RSRP"] == -200]["NetworkMode"].value_counts()

RSRP = -200 : 73


NetworkMode
HSPA+    62
HSUPA     7
LTE       3
UMTS      1
Name: count, dtype: int64

In [5]:
doublons = raca.duplicated(subset=["trace", "Timestamp"])
print("Lignes avec un horodatage déjà vu dans la même trace :", doublons.sum(), f"({doublons.mean():.1%})")

exemple = raca[raca.duplicated(subset=["trace", "Timestamp"], keep=False)]
exemple[["trace", "Timestamp", "RSRP", "SNR", "DL_bitrate"]].head(6)

Lignes avec un horodatage déjà vu dans la même trace : 7968 (4.2%)


,trace,Timestamp,RSRP,SNR,DL_bitrate
0,B_2019.11.28_07.27.57,2019.11.28_07.27.57,-102,8.0,0
1,B_2019.11.28_07.27.57,2019.11.28_07.27.57,-102,8.0,0
85,B_2019.11.28_07.27.57,2019.11.28_07.29.31,-86,21.0,281
86,B_2019.11.28_07.27.57,2019.11.28_07.29.31,-86,21.0,281
100,B_2019.11.28_07.27.57,2019.11.28_07.29.46,-83,20.0,1225
101,B_2019.11.28_07.27.57,2019.11.28_07.29.46,-81,26.0,280


In [6]:
(raca["DL_bitrate"] == 0).groupby(raca["application"]).mean().round(3)

application
Amazon_Prime    0.376
Download        0.038
Netflix         0.737
Name: DL_bitrate, dtype: float64

In [8]:
from src.donnees import nettoyer_raca

propre = nettoyer_raca(raca)
print(propre.shape)
print(propre["generation"].value_counts(), "\n")
propre[["RSRP", "RSRQ", "SNR", "CQI"]].describe().round(1)

(188711, 26)
generation
5G    136403
4G     35212
3G     17005
2G        91
Name: count, dtype: int64 



,RSRP,RSRQ,SNR,CQI
count,171612.0,171451.0,171612.0,171614.0
mean,-91.7,-12.2,3.3,11.0
std,10.8,2.9,7.2,2.9
min,-124.0,-26.0,-19.0,0.0
25%,-100.0,-14.0,-1.0,9.0
50%,-95.0,-12.0,2.0,11.0
75%,-84.0,-10.0,7.0,14.0
max,-48.0,-3.0,35.0,15.0


In [9]:
from src.donnees import reechantillonner_raca

serie = reechantillonner_raca(propre)
print(serie.shape)
print("Secondes complétées :", serie["seconde_completee"].sum(), f"({serie['seconde_completee'].mean():.1%})")
print("Doublons restants :", serie.duplicated(subset=["trace", "temps"]).sum())
serie.groupby("trace")["temps"].diff().dt.total_seconds().value_counts()

(200630, 21)
Secondes complétées : 19887 (9.9%)
Doublons restants : 0


temps
1.0    200547
Name: count, dtype: int64